## Phase 1: Load and design the windows

In [2]:
import numpy as np
import pandas as pd

# Point this at wherever you unzipped ml-25m
PATH = "../data/ml-25m/ratings.csv"

# Specify dtypes up front - 25M rows will eat memory otherwise
ratings = pd.read_csv(PATH, dtype={"userId": "int32", "movieId": "int32",
                                   "rating": "float32", "timestamp": "int64"})

# Unix timestamps are seconds since 1970; convert to real dates
ratings["date"] = pd.to_datetime(ratings["timestamp"], unit="s")

print(ratings.shape)
print("date range:", ratings["date"].min(), "to", ratings["date"].max())
print("users:", ratings["userId"].nunique())
ratings.head()

(25000095, 5)
date range: 1995-01-09 11:46:49 to 2019-11-21 09:15:03
users: 162541


,userId,movieId,rating,timestamp,date
0,1,296,5.0,1147880044,2006-05-17 15:34:04
1,1,306,3.5,1147868817,2006-05-17 12:26:57
2,1,307,5.0,1147868828,2006-05-17 12:27:08
3,1,665,5.0,1147878820,2006-05-17 15:13:40
4,1,899,3.5,1147868510,2006-05-17 12:21:50


In [3]:
# How much activity is there per year? We need a period with plenty of users.
yearly = (ratings.groupby(ratings["date"].dt.year)
          .agg(ratings=("userId", "size"), users=("userId", "nunique")))
print(yearly.to_string())

      ratings  users
date                
1995        3      1
1996  1430093  23028
1997   626202  10014
1998   272099   3315
1999  1059080   8387
2000  1735398  13303
2001  1058750   9300
2002   776654   7098
2003   920295   7674
2004  1048116   7842
2005  1613550  11187
2006  1038458   9167
2007   931432   8472
2008  1018001   9662
2009   810127   8793
2010   792436   8857
2011   676498   8402
2012   635208   7834
2013   515684   7196
2014   478270   6469
2015  1604971  14262
2016  1757440  14270
2017  1689935  13252
2018  1310761  12001
2019  1200634  10601


---
## Phase 2: Build the labels

In [5]:
FEATURE_DAYS    = 365   # how much history we look back at
ELIGIBLE_DAYS   = 90    # must be active in this window to count as a subscriber
LABEL_DAYS      = 90    # how far forward we look to see if they lapsed

def build_dataset(ratings, anchor):
    """Build one (features, labels) snapshot as of `anchor`.

    Everything in the returned features comes from BEFORE the anchor.
    The label comes from AFTER. Nothing crosses the line.
    """
    anchor = pd.Timestamp(anchor)

    # --- the three windows ---
    feat_start   = anchor - pd.Timedelta(days=FEATURE_DAYS)
    elig_start   = anchor - pd.Timedelta(days=ELIGIBLE_DAYS)
    label_end    = anchor + pd.Timedelta(days=LABEL_DAYS)

    history = ratings[(ratings.date >= feat_start) & (ratings.date < anchor)]
    future  = ratings[(ratings.date >= anchor)     & (ratings.date < label_end)]

    # --- who counts as an active subscriber at the anchor? ---
    # Rule: rated at least once in the ELIGIBLE_DAYS before the anchor.
    eligible = history.loc[history.date >= elig_start, "userId"].unique()
    history  = history[history.userId.isin(eligible)]

    # --- label: did they come back at all in the label window? ---
    returned = set(future.loc[future.userId.isin(eligible), "userId"])
    y = pd.Series([0 if u in returned else 1 for u in eligible],
                   index=eligible, name="lapsed")

    return history, y, anchor

In [6]:
# Two snapshots at different points in time.
# Train on the earlier one, test on the later one.

hist_tr, y_tr, anchor_tr = build_dataset(ratings, "2018-01-01")
hist_te, y_te, anchor_te = build_dataset(ratings, "2019-01-01")

print(f"train anchor {anchor_tr.date()}: {len(y_tr):,} users, lapse rate {y_tr.mean():.3f}")
print(f"test  anchor {anchor_te.date()}: {len(y_te):,} users, lapse rate {y_te.mean():.3f}")

train anchor 2018-01-01: 5,045 users, lapse rate 0.503
test  anchor 2019-01-01: 5,160 users, lapse rate 0.511


In [7]:
# Save each snapshot so the next notebook doesn't re-read 25M rows
hist_tr.to_parquet("../data/history_train.parquet")
hist_te.to_parquet("../data/history_test.parquet")
y_tr.to_frame().to_parquet("../data/labels_train.parquet")
y_te.to_frame().to_parquet("../data/labels_test.parquet")

print("saved")

saved
